Collections
Unique record IDs
Dense vectors
Sparse vectors
Multiple named vectors
JSON payloads
Metadata indexes
Insert/update/upsert/delete
Filtering
Persistence
Snapshots
Replication and sharding
HTTP/gRPC server

Qdrant Database
│
├── Collection: company-documents
│     │
│     ├── Point 1
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     ├── Point 2
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     └── Point 3
│
├── Vector index
│     └── HNSW / related retrieval indexes
│
├── Payload index
│     └── category, source, page, user_id...
│
└── Storage
      ├── Memory
      └── Disk

In [7]:
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Sudhe\AppData\Local\Temp\ipykernel_12104\583762284.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Sudhe\anaconda3\envs\sudheer_agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [9]:
COLLECTION_NAME = "sudheer-movies-rag"

In [10]:
# 1. Load PDF

BASE_DIR = Path.cwd()
sudheer_moviest_file = BASE_DIR / "5_Sudheer_Movies_Annual_Report.pdf"
print(sudheer_moviest_file)

loader = PyPDFLoader(sudheer_moviest_file)
pages = loader.load()

print("PDF pages loaded:", len(pages))


c:\Sudheer\Agentic_AI\workspace\python\RAG_Pipeline\5_Sudheer_Movies_Annual_Report.pdf
PDF pages loaded: 4


In [11]:
# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 4


In [12]:
# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "5_Sudheer_Movies_Annual_Report.pdf"

In [13]:
# ---------------------------------------------------
# 2. Embedding model
# ---------------------------------------------------
import os
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)
dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

dimension = len(
    embeddings.embed_query("dimension check")
)


Embedding dimension: 1536


In [ ]:
# # 5. Local Qdrant
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )


In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")


In [15]:
from qdrant_client import QdrantClient, models
client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

In [16]:
# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

In [17]:
# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


In [18]:
# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

In [19]:
# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 4


In [20]:
query = "What is the total gross revenue?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: Director's Message
Dear Stakeholders and Partners,
The 2025-2026 financial year has been a spectacular era for Telugu Cinema, and  Sudheer Telugu
Movies Distribution Company has stood at the forefront of this cinematic revolution. By strategically
acquiring theatrical rights for some of the biggest pan-India and regional tentpoles, we have solidified
our position as a leading distribution house in Andhra Pradesh, Telangana, and key overseas markets.
This year, Tollywood continued to push boundaries with unprecedented scale and storytelling. Our
robust  distribution  strategy  allowed  us  to  successfully  capitalize  on  massive  theatrical  windows,
ensuring  maximum  screen  counts  and  optimal  box  office  recovery.  Our  core  focus  remained  on
nurturing strong relationships with exhibitors and maximizing revenue share across the Nizam, Ceded,
and Andhra territories.
TOTAL GROSS REVENUE
₹845 Cr
THEATRICAL SHARE
₹460 Cr
DISTRIBUTOR PROFIT
₹112 Cr
We acquired 

In [21]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

Director's Message
Dear Stakeholders and Partners,
The 2025-2026 financial year has been a spectacular era for Telugu Cinema, and  Sudheer Telugu
Movies Distribution Company has stood at the forefront of this cinematic revolution. By strategically
acquiring theatrical rights for some of the biggest pan-India and regional tentpoles, we have solidified
our position as a leading distribution house in Andhra Pradesh, Telangana, and key overseas markets.
This year, Tollywood continued to push boundaries with unprecedented scale and storytelling. Our
robust  distribution  strategy  allowed  us  to  successfully  capitalize  on  massive  theatrical  windows,
ensuring  maximum  screen  counts  and  optimal  box  office  recovery.  Our  core  focus  remained  on
nurturing strong relationships with exhibitors and maximizing revenue share across the Nizam, Ceded,
and Andhra territories.
TOTAL GROSS REVENUE
₹845 Cr
THEATRICAL SHARE
₹460 Cr
DISTRIBUTOR PROFIT
₹112 Cr
We acquired phenomenally succes